# 03. Limpieza general base del corpus RTVE 23-F

Este notebook parte del EDA general ya realizado y crea una versión base del corpus con una limpieza textual mínima.

La idea no es preparar todavía datos para un modelo concreto. Cada mini caso podrá necesitar su propia limpieza, su propio tratamiento de texto y su propio pipeline. Por eso, aquí solo se aplican transformaciones generales, simples y trazables.

La limpieza base seguirá estos criterios:

- conservar siempre el texto original en `text_full`;
- crear una nueva columna `text_clean_base`;
- corregir únicamente problemas generales de formato;
- normalizar espacios y saltos de línea;
- evitar transformaciones que dependan de una variable objetivo;
- no hacer TF-IDF, embeddings, escalado, imputación, selección de variables ni balanceo de clases.

El resultado de este notebook será una tabla limpia de partida.

## 1. Carga del corpus descriptivo

Se carga el archivo generado al final del EDA general:

`data/processed/rtve_corpus_eda_descriptive.csv`

Este archivo contiene el texto OCR original, metadatos y algunas métricas descriptivas. En este notebook se utilizará como punto de partida para crear una versión textual limpia, manteniendo siempre el texto original sin modificar.

In [1]:
from pathlib import Path

import pandas as pd

# Localización robusta de la raíz del repositorio
current_path = Path.cwd().resolve()

if (current_path / "data").exists() and (current_path / "notebooks").exists():
    ROOT = current_path
elif (current_path.parent / "data").exists() and (current_path.parent / "notebooks").exists():
    ROOT = current_path.parent
else:
    raise FileNotFoundError(
        "No se ha podido localizar la raíz del repositorio. "
        "Comprueba desde dónde se está ejecutando el notebook."
    )

DATA_PROCESSED = ROOT / "data" / "processed"

input_path = DATA_PROCESSED / "rtve_corpus_eda_descriptive.csv"

df = pd.read_csv(input_path)

print(f"ROOT: {ROOT}")
print(f"Archivo cargado: {input_path}")
print(f"Dimensiones: {df.shape}")

print("\nColumnas disponibles:")
print(df.columns.tolist())

display(df.head())

ROOT: /Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos
Archivo cargado: /Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/data/processed/rtve_corpus_eda_descriptive.csv
Dimensiones: (167, 40)

Columnas disponibles:
['doc_id', 'source', 'source_document_id', 'title', 'pages', 'summary', 'keywords', 'detail_url', 'pdf_url', 'text_full', 'text_length_chars', 'text_length_words', 'extraction_source', 'text_extraction_ok', 'chars_per_page', 'words_per_page', 'chars_per_word', 'flag_long_text_iqr', 'alpha_ratio', 'digit_ratio', 'space_ratio', 'uppercase_ratio', 'empty_line_ratio', 'avg_line_length', 'weird_char_ratio', 'moncloa_id', 'moncloa_section', 'moncloa_subsection', 'final_match_status', 'coverage_moncloa', 'summary_length_chars', 'summary_length_words', 'keywords_length_chars', 'keywords_length_words', 'summary_ends_with_ellipsis', 'keywords_has_date_pattern', 'keyword

,doc_id,source,source_document_id,title,pages,summary,keywords,detail_url,pdf_url,text_full,...,summary_length_chars,summary_length_words,keywords_length_chars,keywords_length_words,summary_ends_with_ellipsis,keywords_has_date_pattern,keywords_has_admin_reference,n_title_years,title_main_year,n_tokens_eda
0,rtve_1860,rtve_buscador,1860,Vista oral 2/81 del Consejo Supremo de Justici...,3,El juicio oral 2/81 celebrado en febrero de 19...,C/SG/2820/20-02-82 DTOR. Vista oral 2/81,https://23fbuscador.rtve.es/document/ocr/1860?...,https://www.rtve.es/contenidos/documentos/23f-...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,...,303,43,40,5,True,True,True,1,1982.0,316
1,rtve_1859,rtve_buscador,1859,Vista oral 2/81 del Consejo Supremo de Justici...,4,Resumen global del documento:\n\nEl documento ...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Sup...,https://23fbuscador.rtve.es/document/ocr/1859?...,https://www.rtve.es/contenidos/documentos/23f-...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,...,303,47,70,9,True,True,True,1,1982.0,531
2,rtve_1858,rtve_buscador,1858,Vista oral 2/81 del Consejo Supremo de Justici...,5,Resumen global del documento:\n\nEl documento ...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Sup...,https://23fbuscador.rtve.es/document/ocr/1858?...,https://www.rtve.es/contenidos/documentos/23f-...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,...,303,49,70,9,True,True,True,1,1982.0,654
3,rtve_1857,rtve_buscador,1857,Vista oral 2/81 del Consejo Supremo de Justici...,6,El documento recoge el desarrollo de la sesión...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Su...,https://23fbuscador.rtve.es/document/ocr/1857?...,https://www.rtve.es/contenidos/documentos/23f-...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,...,303,50,71,9,True,True,True,1,1982.0,900
4,rtve_1856,rtve_buscador,1856,Vista oral 2/81 del Consejo Supremo de Justici...,6,Resumen global del documento sobre la sesión d...,C/SG/3.249/26-02-82 SG Consejo Supremo de Just...,https://23fbuscador.rtve.es/document/ocr/1856?...,https://www.rtve.es/contenidos/documentos/23f-...,C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,...,303,49,58,7,True,True,True,1,1982.0,794


## 2. Criterio de limpieza general

La limpieza general se plantea de forma conservadora. El objetivo no es preparar el corpus para un modelo concreto, sino crear una versión base del texto más ordenada y fácil de reutilizar.

En esta fase no se eliminan documentos, no se balancean clases y no se ajusta ninguna transformación estadística. La limpieza se aplica documento a documento y no depende de ninguna variable objetivo.

Se harán únicamente cambios de formato:

- normalizar saltos de línea;
- eliminar espacios repetidos innecesarios;
- quitar espacios al principio y al final de cada línea;
- reducir bloques excesivos de líneas vacías;
- crear métricas sencillas para comparar el texto original y el texto limpio.

No se hará todavía:

- conversión a minúsculas;
- eliminación de números;
- eliminación de signos de puntuación;
- eliminación de stopwords;
- lematización;
- stemming;
- TF-IDF;
- embeddings;
- train/test split;
- balanceo de clases.

Estas decisiones se dejan para cada mini caso, porque dependen del objetivo concreto del análisis.

In [2]:
import re
import unicodedata

print("=== LIMPIEZA GENERAL BASE DEL TEXTO ===\n")

def clean_text_base(text):
    """
    Limpieza conservadora del texto OCR.

    No elimina contenido semántico.
    No pasa a minúsculas.
    No elimina números, signos, nombres propios ni códigos.
    Solo corrige problemas generales de formato.
    """
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalización unicode básica
    text = unicodedata.normalize("NFKC", text)

    # Normalización de saltos de línea
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Espacios no separables y tabulaciones
    text = text.replace("\xa0", " ")
    text = text.replace("\t", " ")

    # Quitar espacios repetidos dentro de cada línea
    lines = text.split("\n")
    lines = [re.sub(r"[ ]{2,}", " ", line).strip() for line in lines]

    # Reconstruir texto
    text = "\n".join(lines)

    # Reducir bloques de muchas líneas vacías a un máximo de dos saltos seguidos
    text = re.sub(r"\n{3,}", "\n\n", text)

    # Quitar espacios generales al inicio y final
    text = text.strip()

    return text


df_clean = df.copy()

df_clean["text_clean_base"] = df_clean["text_full"].apply(clean_text_base)

df_clean["text_clean_length_chars"] = df_clean["text_clean_base"].str.len()
df_clean["text_clean_length_words"] = df_clean["text_clean_base"].str.split().str.len()

df_clean["clean_chars_difference"] = (
    df_clean["text_length_chars"] - df_clean["text_clean_length_chars"]
)

df_clean["clean_words_difference"] = (
    df_clean["text_length_words"] - df_clean["text_clean_length_words"]
)

print("Dimensiones tras crear texto limpio:")
print(df_clean.shape)

print("\nComparación de longitud antes y después de la limpieza:")
display(
    df_clean[
        [
            "text_length_chars",
            "text_clean_length_chars",
            "clean_chars_difference",
            "text_length_words",
            "text_clean_length_words",
            "clean_words_difference",
        ]
    ].describe().T
)

print("\nDocumentos con mayor diferencia en caracteres tras la limpieza:")
display(
    df_clean[
        [
            "doc_id",
            "title",
            "text_length_chars",
            "text_clean_length_chars",
            "clean_chars_difference",
            "text_length_words",
            "text_clean_length_words",
            "clean_words_difference",
        ]
    ]
    .sort_values("clean_chars_difference", ascending=False)
    .head(10)
)

=== LIMPIEZA GENERAL BASE DEL TEXTO ===

Dimensiones tras crear texto limpio:
(167, 45)

Comparación de longitud antes y después de la limpieza:


,count,mean,std,min,25%,50%,75%,max
text_length_chars,167.0,11675.892216,37822.711182,455.0,1495.5,3529.0,10524.0,453130.0
text_clean_length_chars,167.0,11653.682635,37799.925011,455.0,1495.5,3529.0,10454.0,453136.0
clean_chars_difference,167.0,22.209581,249.061173,-6.0,0.0,0.0,0.0,3214.0
text_length_words,167.0,2075.443114,7731.242472,72.0,238.5,579.0,1771.5,95293.0
text_clean_length_words,167.0,2075.443114,7731.242472,72.0,238.5,579.0,1771.5,95293.0
clean_words_difference,167.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0



Documentos con mayor diferencia en caracteres tras la limpieza:


,doc_id,title,text_length_chars,text_clean_length_chars,clean_chars_difference,text_length_words,text_clean_length_words,clean_words_difference
74,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo de...",56382,53168,3214,11080,11080,0
163,rtve_1697,"""Documentación con una presunta planificación ...",28585,28434,151,4882,4882,0
84,rtve_1776,D.31._AGA-83-09301_exp._5,10398,10258,140,1926,1926,0
36,rtve_1824,Vista oral 2/81 del Consejo Supremo de Justici...,6962,6896,66,1279,1279,0
159,rtve_1701,Télex interiores y de agencias recibidos en 2ª...,119495,119453,42,19857,19857,0
157,rtve_1703,Semestral de la amenaza interior (10 de febrer...,56898,56882,16,8714,8714,0
141,rtve_1719,RESERVADO: Informe de Asesoría Jurídica Genera...,2790,2776,14,494,494,0
136,rtve_1724,RESERVADO: oficio dando cuenta toma de declara...,1030,1017,13,193,193,0
123,rtve_1737,SECRETO: comunicando sanción a consejeros del ...,1217,1205,12,218,218,0
143,rtve_1717,RESERVADO: parte por abandono de destino del C...,4271,4259,12,715,715,0


**Lectura del bloque**

Se crea una nueva columna `text_clean_base`, pero se conserva intacto el texto original en `text_full`.

La comparación antes y después muestra que no cambia el número de palabras en ningún documento. Esto es importante porque indica que la limpieza no está eliminando contenido textual, sino corrigiendo principalmente aspectos de formato: espacios, tabulaciones, saltos de línea o caracteres normalizados.

La mayoría de documentos no cambia o cambia muy poco. El caso con mayor diferencia en caracteres es `rtve_1786`, con 3.214 caracteres menos, pero mantiene exactamente el mismo número de palabras. Por tanto, la diferencia parece estar relacionada con formato y no con pérdida de contenido.

Esta versión limpia debe entenderse como una base común para trabajar más cómodamente con el corpus.

## 3. Validación de la limpieza base

Antes de guardar el resultado, se comprueba que la limpieza no haya introducido problemas básicos: pérdida de documentos, textos vacíos, duplicados o cambios fuertes no esperados.

In [3]:
print("=== VALIDACIÓN DE LA LIMPIEZA BASE ===\n")

print("Dimensiones originales:")
print(df.shape)

print("\nDimensiones tras limpieza:")
print(df_clean.shape)

print("\nDuplicados por doc_id:")
print(df_clean["doc_id"].duplicated().sum())

print("\nTextos originales vacíos:")
print(df_clean["text_full"].fillna("").str.strip().eq("").sum())

print("\nTextos limpios vacíos:")
print(df_clean["text_clean_base"].fillna("").str.strip().eq("").sum())

print("\nDocumentos con cambio en número de palabras:")
n_word_changes = (df_clean["clean_words_difference"] != 0).sum()
print(n_word_changes)

print("\nDistribución del cambio en caracteres:")
display(
    df_clean["clean_chars_difference"]
    .describe()
    .to_frame()
    .T
)

print("\nDocumentos donde el texto limpio tiene más caracteres que el original:")
display(
    df_clean.loc[
        df_clean["clean_chars_difference"] < 0,
        [
            "doc_id",
            "title",
            "text_length_chars",
            "text_clean_length_chars",
            "clean_chars_difference",
        ],
    ]
    .sort_values("clean_chars_difference")
)

print("\nDocumentos con mayor reducción relativa de caracteres:")
df_clean["clean_chars_reduction_pct"] = (
    df_clean["clean_chars_difference"] / df_clean["text_length_chars"] * 100
)

display(
    df_clean[
        [
            "doc_id",
            "title",
            "text_length_chars",
            "text_clean_length_chars",
            "clean_chars_difference",
            "clean_chars_reduction_pct",
        ]
    ]
    .sort_values("clean_chars_reduction_pct", ascending=False)
    .head(10)
)

=== VALIDACIÓN DE LA LIMPIEZA BASE ===

Dimensiones originales:
(167, 40)

Dimensiones tras limpieza:
(167, 45)

Duplicados por doc_id:
0

Textos originales vacíos:
0

Textos limpios vacíos:
0

Documentos con cambio en número de palabras:
0

Distribución del cambio en caracteres:


,count,mean,std,min,25%,50%,75%,max
clean_chars_difference,167.0,22.209581,249.061173,-6.0,0.0,0.0,0.0,3214.0



Documentos donde el texto limpio tiene más caracteres que el original:


,doc_id,title,text_length_chars,text_clean_length_chars,clean_chars_difference
161,rtve_1699,Transcripción de cintas grabadas con conversac...,453130,453136,-6
71,rtve_1789,"""Nota """"Posible golpe de estado"""" (sin firma).""",6223,6227,-4



Documentos con mayor reducción relativa de caracteres:


,doc_id,title,text_length_chars,text_clean_length_chars,clean_chars_difference,clean_chars_reduction_pct
74,rtve_1786,"""Juicio del 23-F: acotaciones al desarrollo de...",56382,53168,3214,5.700401
84,rtve_1776,D.31._AGA-83-09301_exp._5,10398,10258,140,1.346413
136,rtve_1724,RESERVADO: oficio dando cuenta toma de declara...,1030,1017,13,1.262136
99,rtve_1761,D.17._AGMAE_R40201_Exp._215,982,971,11,1.120163
123,rtve_1737,SECRETO: comunicando sanción a consejeros del ...,1217,1205,12,0.986031
36,rtve_1824,Vista oral 2/81 del Consejo Supremo de Justici...,6962,6896,66,0.948003
81,rtve_1779,"""Nota de la Brigada de Información Interior: E...",1197,1189,8,0.668338
163,rtve_1697,"""Documentación con una presunta planificación ...",28585,28434,151,0.528249
141,rtve_1719,RESERVADO: Informe de Asesoría Jurídica Genera...,2790,2776,14,0.501792
143,rtve_1717,RESERVADO: parte por abandono de destino del C...,4271,4259,12,0.280965


## 4. Revisión de columnas

Antes de guardar la base limpia, se revisan las columnas disponibles para decidir cuáles merece la pena conservar.

La idea es no arrastrar variables que solo sirvieron para el EDA o para validar la extracción. Algunas columnas pueden ser constantes, tener valores repetidos en todos los documentos o ser útiles solo como diagnóstico temporal.

Esta revisión no modifica todavía la tabla. Solo ayuda a decidir una selección final de columnas más limpia y razonable.

In [4]:
print("=== REVISIÓN DE COLUMNAS DISPONIBLES ===\n")

column_audit = []

for col in df_clean.columns:
    n_null = df_clean[col].isna().sum()
    n_unique = df_clean[col].nunique(dropna=False)
    
    # Ejemplos compactos de valores
    sample_values = (
        df_clean[col]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .head(3)
        .tolist()
    )
    
    column_audit.append({
        "column": col,
        "dtype": str(df_clean[col].dtype),
        "n_null": n_null,
        "n_unique": n_unique,
        "sample_values": sample_values,
    })

column_audit_df = pd.DataFrame(column_audit)

print("Resumen general de columnas:")
display(column_audit_df)

print("\nColumnas con un único valor:")
display(
    column_audit_df[column_audit_df["n_unique"] == 1]
    .sort_values("column")
)

print("\nColumnas con pocos valores distintos:")
display(
    column_audit_df[
        (column_audit_df["n_unique"] > 1) &
        (column_audit_df["n_unique"] <= 10)
    ]
    .sort_values("n_unique")
)

print("\nPrimeras filas de columnas principales:")
display(
    df_clean[
        [
            "doc_id",
            "title",
            "pages",
            "source",
            "extraction_source",
            "text_extraction_ok",
            "moncloa_section",
            "moncloa_subsection",
            "coverage_moncloa",
            "summary_ends_with_ellipsis",
            "keywords_has_date_pattern",
            "keywords_has_admin_reference",
        ]
    ].head(10)
)

=== REVISIÓN DE COLUMNAS DISPONIBLES ===

Resumen general de columnas:


,column,dtype,n_null,n_unique,sample_values
0,doc_id,object,0,167,"[rtve_1860, rtve_1859, rtve_1858]"
1,source,object,0,1,[rtve_buscador]
2,source_document_id,int64,0,167,"[1860, 1859, 1858]"
3,title,object,0,159,[Vista oral 2/81 del Consejo Supremo de Justic...
4,pages,int64,0,25,"[3, 4, 5]"
5,summary,object,0,167,[El juicio oral 2/81 celebrado en febrero de 1...
6,keywords,object,0,166,"[C/SG/2820/20-02-82 DTOR. Vista oral 2/81, C/S..."
7,detail_url,object,0,167,[https://23fbuscador.rtve.es/document/ocr/1860...
8,pdf_url,object,0,167,[https://www.rtve.es/contenidos/documentos/23f...
9,text_full,object,0,167,[C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA...



Columnas con un único valor:


,column,dtype,n_null,n_unique,sample_values
44,clean_words_difference,int64,0,1,[0]
12,extraction_source,object,0,1,[rtve_detail_html_pre]
1,source,object,0,1,[rtve_buscador]
34,summary_ends_with_ellipsis,bool,0,1,[True]
30,summary_length_chars,int64,0,1,[303]
13,text_extraction_ok,bool,0,1,[True]



Columnas con pocos valores distintos:


,column,dtype,n_null,n_unique,sample_values
17,flag_long_text_iqr,bool,0,2,"[False, True]"
36,keywords_has_admin_reference,bool,0,2,"[True, False]"
35,keywords_has_date_pattern,bool,0,2,"[True, False]"
29,coverage_moncloa,bool,0,2,"[True, False]"
28,final_match_status,object,12,3,"[high_confidence_match, manual_match]"
37,n_title_years,int64,0,3,"[1, 2, 0]"
26,moncloa_section,object,12,4,"[defensa, interior, exteriores]"
27,moncloa_subsection,object,55,5,"[cni, archivo, policia]"
38,title_main_year,float64,78,7,"[1982.0, 1981.0, 1983.0]"



Primeras filas de columnas principales:


,doc_id,title,pages,source,extraction_source,text_extraction_ok,moncloa_section,moncloa_subsection,coverage_moncloa,summary_ends_with_ellipsis,keywords_has_date_pattern,keywords_has_admin_reference
0,rtve_1860,Vista oral 2/81 del Consejo Supremo de Justici...,3,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
1,rtve_1859,Vista oral 2/81 del Consejo Supremo de Justici...,4,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
2,rtve_1858,Vista oral 2/81 del Consejo Supremo de Justici...,5,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
3,rtve_1857,Vista oral 2/81 del Consejo Supremo de Justici...,6,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
4,rtve_1856,Vista oral 2/81 del Consejo Supremo de Justici...,6,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
5,rtve_1855,Información integrada (11 de marzo de 1982).,2,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
6,rtve_1854,Información integrada (16 de marzo de 1982).,2,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
7,rtve_1853,Vista oral 2/81 del Consejo Supremo de Justici...,10,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
8,rtve_1852,Vista oral 2/81 del Consejo Supremo de Justici...,8,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True
9,rtve_1851,Vista oral 2/81 del Consejo Supremo de Justici...,5,rtve_buscador,rtve_detail_html_pre,True,defensa,cni,True,True,True,True


**Lectura del bloque**

La revisión de columnas muestra que algunas variables fueron útiles durante el EDA, pero no aportan valor suficiente para la base limpia final.

Se eliminan columnas constantes como `source`, `extraction_source`, `text_extraction_ok`, `summary_length_chars`, `summary_ends_with_ellipsis` y `clean_words_difference`, ya que tienen el mismo valor en todos los documentos.

También se descartan métricas intermedias del EDA, como ratios por página, longitud media de línea, líneas vacías o diferencias de limpieza. Estas variables ayudaron a entender el corpus, pero no son necesarias en una tabla base común.

La base limpia conservará identificación, trazabilidad, texto original, texto limpio, longitudes básicas, metadatos institucionales y algunos indicadores generales de calidad textual. Esta tabla seguirá sin ser una matriz de entrenamiento: cada mini caso deberá decidir qué columnas usa y cuáles excluye para evitar leakage.

In [8]:
print("=== SELECCIÓN FINAL DE COLUMNAS PARA LA BASE LIMPIA ===\n")

columns_clean_base = [
    # Identificación y trazabilidad
    "doc_id",
    "source_document_id",
    "title",
    "pages",
    "detail_url",
    "pdf_url",

    # Metadatos textuales
    "summary",
    "keywords",

    # Texto original y texto limpio
    "text_full",
    "text_clean_base",

    # Longitudes básicas
    "text_length_chars",
    "text_length_words",
    "text_clean_length_chars",
    "text_clean_length_words",

    # Cobertura institucional
    "moncloa_id",
    "moncloa_section",
    "moncloa_subsection",
    "final_match_status",
    "coverage_moncloa",

    # Indicadores generales de calidad
    "alpha_ratio",
    "digit_ratio",
    "uppercase_ratio",
    "weird_char_ratio",

    # Información temporal básica desde título
    "n_title_years",
    "title_main_year",
]

missing_columns = [col for col in columns_clean_base if col not in df_clean.columns]

if missing_columns:
    raise ValueError(f"Faltan columnas esperadas: {missing_columns}")

df_clean_base = df_clean[columns_clean_base].copy()

print("Dimensiones de df_clean completo:")
print(df_clean.shape)

print("\nDimensiones de df_clean_base:")
print(df_clean_base.shape)

print("\nColumnas conservadas:")
print(df_clean_base.columns.tolist())

display(df_clean_base.head())

=== SELECCIÓN FINAL DE COLUMNAS PARA LA BASE LIMPIA ===

Dimensiones de df_clean completo:
(167, 46)

Dimensiones de df_clean_base:
(167, 25)

Columnas conservadas:
['doc_id', 'source_document_id', 'title', 'pages', 'detail_url', 'pdf_url', 'summary', 'keywords', 'text_full', 'text_clean_base', 'text_length_chars', 'text_length_words', 'text_clean_length_chars', 'text_clean_length_words', 'moncloa_id', 'moncloa_section', 'moncloa_subsection', 'final_match_status', 'coverage_moncloa', 'alpha_ratio', 'digit_ratio', 'uppercase_ratio', 'weird_char_ratio', 'n_title_years', 'title_main_year']


,doc_id,source_document_id,title,pages,detail_url,pdf_url,summary,keywords,text_full,text_clean_base,...,moncloa_section,moncloa_subsection,final_match_status,coverage_moncloa,alpha_ratio,digit_ratio,uppercase_ratio,weird_char_ratio,n_title_years,title_main_year
0,rtve_1860,1860,Vista oral 2/81 del Consejo Supremo de Justici...,3,https://23fbuscador.rtve.es/document/ocr/1860?...,https://www.rtve.es/contenidos/documentos/23f-...,El juicio oral 2/81 celebrado en febrero de 19...,C/SG/2820/20-02-82 DTOR. Vista oral 2/81,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,C/SG/2820/20-02-82\nDTOR.\n\nNOTA INFORMATIVA\...,...,defensa,cni,high_confidence_match,True,0.777834,0.013726,0.147386,0.000000,1,1982.0
1,rtve_1859,1859,Vista oral 2/81 del Consejo Supremo de Justici...,4,https://23fbuscador.rtve.es/document/ocr/1859?...,https://www.rtve.es/contenidos/documentos/23f-...,Resumen global del documento:\n\nEl documento ...,C/SG/2896/22-02-82 Vista oral 2/81 Consejo Sup...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,C/SG/2896/22-02-82\n\n# NOTA INFORMATIVA\n\nAS...,...,defensa,cni,high_confidence_match,True,0.781985,0.009506,0.195895,0.000156,1,1982.0
2,rtve_1858,1858,Vista oral 2/81 del Consejo Supremo de Justici...,5,https://23fbuscador.rtve.es/document/ocr/1858?...,https://www.rtve.es/contenidos/documentos/23f-...,Resumen global del documento:\n\nEl documento ...,C/SG/2992/24-02-82 Vista Oral 2/81 Consejo Sup...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,C/SG/2992/24-02-82\n\n# NOTA INFORMATIVA\n\nAS...,...,defensa,cni,high_confidence_match,True,0.784920,0.011487,0.124085,0.000611,1,1982.0
3,rtve_1857,1857,Vista oral 2/81 del Consejo Supremo de Justici...,6,https://23fbuscador.rtve.es/document/ocr/1857?...,https://www.rtve.es/contenidos/documentos/23f-...,El documento recoge el desarrollo de la sesión...,C/SG/3.081/25-02-82 Vista Oral 2/81 Consejo Su...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,C/SG/3.081/25-02-82\n\n# NOTA INFORMATIVA\n\nA...,...,defensa,cni,high_confidence_match,True,0.789257,0.008250,0.128167,0.000538,1,1982.0
4,rtve_1856,1856,Vista oral 2/81 del Consejo Supremo de Justici...,6,https://23fbuscador.rtve.es/document/ocr/1856?...,https://www.rtve.es/contenidos/documentos/23f-...,Resumen global del documento sobre la sesión d...,C/SG/3.249/26-02-82 SG Consejo Supremo de Just...,C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,C/SG/3.249/26-02-82\nSG\n\n# NOTA INFORMATIVA\...,...,defensa,cni,high_confidence_match,True,0.786053,0.008593,0.066097,0.000691,1,1982.0


**Lectura del bloque**

La base limpia se reduce de 46 a 26 columnas. Se conservan las variables necesarias para identificar cada documento, mantener trazabilidad, acceder al texto original, trabajar con el texto limpio y disponer de algunos indicadores generales de calidad.

Se eliminan columnas constantes o puramente auxiliares del EDA, como `source`, `extraction_source`, `text_extraction_ok`, `summary_length_chars`, `summary_ends_with_ellipsis` o `clean_words_difference`. También se descartan métricas intermedias que sirvieron para explorar el corpus, pero que no son necesarias en una base común.

La tabla resultante no es una matriz final de entrenamiento. Es una base ordenada sobre la que cada mini caso podrá decidir su propio tratamiento. Las variables institucionales se conservan como metadatos, pero no deben usarse como predictores si el objetivo del mini caso es clasificar documentos por procedencia institucional. Del mismo modo, los indicadores de calidad textual deben entenderse como variables de diagnóstico, no como features obligatorias para modelos.

## 5. Guardado de la base limpia

Por último, se guarda la tabla limpia en `data/processed` y una copia en `outputs/tables`.

El archivo generado será la base común para los siguientes notebooks. Aun así, cada mini caso deberá construir su propio dataset de trabajo y aplicar sus transformaciones específicas dentro del flujo correcto.

In [9]:
print("=== GUARDADO DE LA BASE LIMPIA ===\n")

processed_dir = ROOT / "data" / "processed"
outputs_tables_dir = ROOT / "outputs" / "tables"

processed_dir.mkdir(parents=True, exist_ok=True)
outputs_tables_dir.mkdir(parents=True, exist_ok=True)

clean_base_path = processed_dir / "rtve_corpus_clean_base.csv"
clean_base_output_path = outputs_tables_dir / "rtve_corpus_clean_base.csv"

df_clean_base.to_csv(clean_base_path, index=False)
df_clean_base.to_csv(clean_base_output_path, index=False)

print(f"Base limpia guardada en:\n{clean_base_path}")
print(f"\nCopia guardada en outputs:\n{clean_base_output_path}")
print(f"\nDimensiones finales: {df_clean_base.shape}")

print("\nColumnas finales:")
print(df_clean_base.columns.tolist())

=== GUARDADO DE LA BASE LIMPIA ===

Base limpia guardada en:
/Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/data/processed/rtve_corpus_clean_base.csv

Copia guardada en outputs:
/Users/gabrielrezola/TrabajoML/RepositoriosGitHub/Proyecto-Machine-Learning-UNAV---Bayesianos-de-los-Ca-dos/outputs/tables/rtve_corpus_clean_base.csv

Dimensiones finales: (167, 25)

Columnas finales:
['doc_id', 'source_document_id', 'title', 'pages', 'detail_url', 'pdf_url', 'summary', 'keywords', 'text_full', 'text_clean_base', 'text_length_chars', 'text_length_words', 'text_clean_length_chars', 'text_clean_length_words', 'moncloa_id', 'moncloa_section', 'moncloa_subsection', 'final_match_status', 'coverage_moncloa', 'alpha_ratio', 'digit_ratio', 'uppercase_ratio', 'weird_char_ratio', 'n_title_years', 'title_main_year']
